# 02 Data Warehouse Design

## Business Questions

### Overall Fatality Analysis
- What are the trends in road accident fatalities by year and month?
- Which days of the week have the highest number of fatalities?
- Which season has the highest average number of fatalities per year?

### Geographic Analysis
- What is the distribution of fatalities across regions?
- Which provinces and districts have the highest numbers of fatalities?
- Which provinces record the highest number of fatalities each year?

### Vehicle Analysis
- Which vehicle types are associated with the highest number of fatalities?

### Demographic Analysis
- Which age groups experience the highest number of fatalities?
- Which gender experiences the highest number of fatalities?

## Data Integration Requirements

### Geographic Reference Data

| Source | Purpose |
|---------|---------|
| Thailand Subdistrict Geographic Dataset | Standardize province, district, and subdistrict names, and enrich geographic attributes. |
| Province Abbreviation Reference | Map province abbreviations to official province names and ensure consistency across years. |

**Sources:**
- [ข้อมูลพิกัดตำบล](https://gdcatalognhic.nha.co.th/dataset/tambon/resource/d65b115f-8ee6-4381-8d6c-6efff538a1b7) 
- [รายชื่ออักษรย่อของจังหวัดในประเทศไทย](https://th.wikipedia.org/wiki/%E0%B8%A3%E0%B8%B2%E0%B8%A2%E0%B8%8A%E0%B8%B7%E0%B9%88%E0%B8%AD%E0%B8%AD%E0%B8%B1%E0%B8%81%E0%B8%A9%E0%B8%A3%E0%B8%A2%E0%B9%88%E0%B8%AD%E0%B8%82%E0%B8%AD%E0%B8%87%E0%B8%88%E0%B8%B1%E0%B8%87%E0%B8%AB%E0%B8%A7%E0%B8%B1%E0%B8%94%E0%B9%83%E0%B8%99%E0%B8%9B%E0%B8%A3%E0%B8%B0%E0%B9%80%E0%B8%97%E0%B8%A8%E0%B9%84%E0%B8%97%E0%B8%A2)

### Date Dimension Reference

| Source | Purpose |
|---------|---------|
| DimDate Reference Table | Generate the date dimension and enrich calendar attributes. |

**Source:**
- [dimdates.com](https://dimdates.com/)

**Additional Enrichment:**
- Add **Buddhist Era (B.E.)** year.
- Derive **Season** from the calendar date.

### ICD-10 code and referencing

| Source | Purpose |
|---------|---------|
| National Center for Health Statistics | Provides standardized ICD-10 classification codes and definitions for causes of death used to build and validate dim_cause_of_death |

**Source:**
- [ICD-10 Classification (NCHS, CDC)](https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Publications/ICD10/)

## Fact Identification

**Name:** fact_fatality

| Column Name | Data Type | Nullable | Description | Business Rule |
|-------------|-----------|----------|-------------|---------------|
| death_record_key | INT | No | Surrogate key generated by system to uniquely identify each deceased record | Primary key, Unique, not null |
| death_date_key | INT | No | Foreign key referencing dim_date, represents date of death | Foreign key with dim_date(id), not null |
| demographic_key | INT | No | Foreign key referencing dim_demographic, represents a person's demographic profile, including age group and gender | Foreign key with dim_demographic(id), not null |
| nationality_key | INT | No | Foreign key referencing dim_nationality, representing a person's nationality based on the ISO 3166-1 standard. Unknown or unmapped nationalities are mapped to the 'Unknown' member in dim_nationality | Foreign key with dim_nationality(id), not null |
| living_location_key | INT | No | Foreign key referencing dim_location, represents person's place of residence | Foreign key with dim_location(id), not null |
| death_location_key | INT | No | Foreign key referencing dim_location, represents location where the person died | Foreign key with dim_location(id), not null |
| vehicle_key | INT | No | Foreign key referencing dim_vehicle, representing the vehicle involved in the accident | Foreign key with dim_vehicle(id), not null |
| cause_key | INT | No | Foreign key referencing dim_cause_of_death, representing the cause of death based on the ICD-10 classification | Foreign key with dim_cause_of_death(id) |
| fatality_count | INT | No | Additive measure representing the number of fatalities | Default value = 1, not null |

**Grain:** One row in fact_fatality represents one deceased person resulting from a road traffic accident.


## Dimension Identification

**Name:** dim_date

| Column Name | Data Type | Nullable | Description | Business Rule |
|-------------|-----------|----------|-------------|---------------|
| id | INT | No | date identifier | Primary key, not null |
| date | DATE | No | Full calendar date associated with the date dimension record| Must be in yyyy-MM-dd format (e.g., 2000-01-01), not null. |
| date_description | string | No |Description of a date | e.g. 1 Jan 2000, not null. |
| day_fullname | string | No | Full name of a day | e.g. Saturday, not null |
| day_shortname | string | No | Short name of a day | e.g. Sat, not null |
| month_fullname | string | No | Full name of a month | e.g. January, not null |
| month_shortname | string | No | Short name of a month | e.g. Jan, not null |
| season | string | No | Season based on Thai meteorological seasons | e.g. Rainy season, not null |
| calendar_day | INT | No | The day number in the calendar year | not null |
| calendar_day_in_week | INT | No | The day number in the week. | not null |
| calendar_month | INT | No | The calendar month number in the year. | not null |
| calendar_day_in_month | INT | No | The day number in the calendar month. | not null |
| calendar_quarter | INT | No | The calendar quarter number in the year. | not null |
| calendar_day_in_quarter | INT | No |	The day number in the calendar quarter. | not null |
| calendar_day_in_season | INT | No |	The day number in the season. | not null |
| calendar_year | INT | No | The calendar year number. | not null |
| calendar_year_be | INT | No | The calendar year number in B.E. | not null |


**Name:** dim_demographic

| Column Name | Data Type | Nullable | Description | Business Rule |
|-------------|-----------|----------|-------------|---------------|
| id | INT | No | demographic identifier, -1 is unknown value | Primary key, not null |
| age_from | INT | Yes | start age of age group | - |
| age_to | INT | Yes | Til age of age group | - |
| age_group | string | No | Name group age group | not null |
| sex | string | No | Gender | not null |

**Name:** dim_nationality

| Column Name | Data Type | Nullable | Description | Business Rule |
|-------------|-----------|----------|-------------|---------------|
| id | INT | No | Nationality identifier, 99 is unknown value. Base on iso 3166-1 | Primary key, not null |
| nationality_name | string | No | Name of nationality | not null |
| nationality_2_alpha | string | No | 2 alphabet name | not null |
| nationality_3_alpha | string | No | 3 alphabet name | not null |

**Name:** dim_location

| Column Name | Data Type | Nullable | Description | Business Rule |
|-------------|-----------|----------|-------------|---------------|
| id | INT | No | location identifier | Primary Key, not null |
| subdistrict | string | No | Subdistrict name, store N/A as Unknown | not null |
| district | string | No | District name, store N/A as Unknown | not null |
| province | string | No | Province name, store N/A as Unknown | not null |
| subdistrict_th | string | No | Subdistrict name in thai, store N/A as Unknown | not null |
| district_th | string | No | District name in thai, store N/A as Unknown | not null |
| province_th | string | No | Province name in thai, store N/A as Unknown | not null |
| province_short_name_th | string | No | Province name in thai, store N/A as Unknown | not null |
| lat | float | Yes | latitude | - |
| lon | float | Yes | longitude | - |

**Name:** dim_vehicle

| Column Name | Data Type | Nullable | Description | Business Rule |
|-------------|-----------|----------|-------------|---------------|
| id | INT | No | vehicle identifier | Primary Key, not null |
| vehicle_type_th | string | No | vehicle type, N/A with unknown | - |
| vehicle_type_en | string | No | vehicle type, N/A with unknown | - |

**Name:** dim_cause_of_death

| Column Name | Data Type | Nullable | Description | Business Rule |
|-------------|-----------|----------|-------------|---------------|
| id | INT | No | cause identifier | Primary Key, not null |
| code | string | No | ICD-10 code, 1 field for unknown | not null |
| description | string | No | ICD-10 description | - |

## Star Schema Design